In [1]:
import os 
import streamlit as st
import pickle
import time 
import langchain
from langchain_openai import ChatOpenAI
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_classic.chains import RetrievalQAWithSourcesChain
from langchain_classic.chains.qa_with_sources.loading import load_qa_with_sources_chain
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.document_loaders import UnstructuredURLLoader
from langchain_google_genai import GoogleGenerativeAIEmbeddings

c:\Users\dheekonda.srinivasar\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1
c:\Users\dheekonda.srinivasar\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
os.environ["GOOGLE_API_KEY"] = "***********"

In [ ]:
llm = ChatGoogleGenerativeAI(
    model="gemini-2.0-flash", # Note: gemini-2.5-flash isn't out yet; 1.5-flash is current.
  
    max_retries=5,        # Automatically retry if it fails
    timeout=60,
    flatten_pdf_metadata=True
)



Unexpected argument 'flatten_pdf_metadata' provided to ChatGoogleGenerativeAI. Did you mean: 'default_metadata'?
C:\Users\dheekonda.srinivasar\AppData\Roaming\Python\Python314\site-packages\IPython\core\interactiveshell.py:3701: UserWarning: WARNING! flatten_pdf_metadata is not default parameter.
                flatten_pdf_metadata was transferred to model_kwargs.
                Please confirm that flatten_pdf_metadata is what you intended.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [4]:
loaders = UnstructuredURLLoader(urls=[
    "https://www.moneycontrol.com/news/business/markets/wall-street-rises-as-tesla-soars-on-ai-optimism-11351111.html",
    "https://www.moneycontrol.com/news/business/tata-motors-launches-punch-icng-price-starts-at-rs-7-1-lakh-11098751.html"
])
data = loaders.load()


In [5]:
len(data)

2

(2) Split data to create chunks

In [6]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 200
)

In [7]:
docs = text_splitter.split_documents(data)


In [8]:
len(docs)

19

In [83]:
docs[0]

Document(metadata={'source': 'https://www.moneycontrol.com/news/business/markets/wall-street-rises-as-tesla-soars-on-ai-optimism-11351111.html'}, page_content='English\n\nHindi\n\nGujarati\n\nSpecials\n\nMy Alerts\n\nGo Ad-Free\n\nHello, Login\n\nHello, Login\n\nLog-inor Sign-Up\n\nMy Account\n\nMy Profile\n\nMy Portfolio\n\nMy Watchlist\n\nMy Alerts\n\nMy Messages\n\nPrice Alerts\n\nMy Profile\n\nMy PRO\n\nMy Portfolio\n\nMy Watchlist\n\nMy Alerts\n\nMy Messages\n\nPrice Alerts\n\nLogout\n\nLoans up to ₹50 LAKHS\n\nFixed Deposits\n\nCredit CardsLifetime Free\n\nCredit Score\n\nLoan against MFs\n\nChat with Us\n\nDownload App\n\nFollow us on:\n\nNetwork 18\n\n>->MC_ENG_DESKTOP/MC_ENG_NEWS/MC_ENG_MARKETS_AS/MC_ENG_ROS_NWS_MKTS_AS_ATF_728\n\nMoneycontrol\n\nGo PRO NowPRO\n\nMoneycontrol PRO\n\nAdvertisement\n\nRemove Ad\n\nBusiness\n\nMarkets\n\nStocks\n\nEconomy\n\nCompanies\n\nTrends\n\nIPO\n\nOpinion\n\nEV Special\n\nEco Pulse\n\nSwing Trading 101\n\nSwing Trading 101\n\nTrending Topi

(3) Create embeddings for these chunks and save them to FAISS index

In [9]:
# Create the embeddings of the chunks using openAIEmbeddings
embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-2-preview")

In [10]:
# Pass the documents and embeddings inorder to create FAISS vector index
vectorindex_google = FAISS.from_documents(docs, embeddings)

In [11]:
## to save
vectorindex_google.save_local("google_faiss_index")

In [20]:
# 2. Load the raw data
new_db = FAISS.load_local("google_faiss_index", embeddings, allow_dangerous_deserialization=True)

(4) Retrieve similar embeddings for a given question and call LLM to retrieve final answer


In [88]:
chain = RetrievalQAWithSourcesChain.from_llm(llm=llm, retriever=new_db.as_retriever())
chain

RetrievalQAWithSourcesChain(verbose=False, combine_documents_chain=MapReduceDocumentsChain(verbose=False, llm_chain=LLMChain(verbose=False, prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='Use the following portion of a long document to see if any of the text is relevant to answer the question.\nReturn any relevant text verbatim.\n{context}\nQuestion: {question}\nRelevant text, if any:'), llm=ChatGoogleGenerativeAI(profile={'max_input_tokens': 1048576, 'max_output_tokens': 8192, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': True, 'pdf_inputs': True, 'video_inputs': True, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'structured_output': True, 'image_url_inputs': True, 'image_tool_message': True, 'tool_choice': True}, google_api_key=SecretStr('**********'), model='gemini-2.0-flash', max_retries=5, timeout=60.0, client

In [93]:
Query = "what is the price of Tiago iCNG?"
response = chain.invoke({"question": Query})
print(response["answer"])

ChatGoogleGenerativeAIError: Error calling model 'gemini-2.0-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\nPlease retry in 10.632057222s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url

In [ ]:
import os
from groq import Groq

# Initialize the client with your key
client = Groq(
   
)

def get_car_info(query):
    try:
        chat_completion = client.chat.completions.create(
            # Using Llama 3.3 70B (very smart, similar to GPT-4o)
            model="llama-3.3-70b-versatile",
            messages=[
                {
                    "role": "user",
                    "content": query,
                }
            ],
        )
        return chat_completion.choices[0].message.content
    except Exception as e:
        return f"An error occurred: {e}"

# Your specific query
query = "what is the price of Tiago iCNG?"
response = get_car_info(query)
print(response)

The price of the Tiago iCNG varies depending on the variant and location. As of my knowledge cutoff in 2023, the prices for the Tiago iCNG in India are as follows:

* Tiago iCNG XE: around ₹7.40 lakhs (ex-showroom)
* Tiago iCNG XM: around ₹7.80 lakhs (ex-showroom)
* Tiago iCNG XZ: around ₹8.30 lakhs (ex-showroom)
* Tiago iCNG XZ+ (Dual Tone): around ₹8.70 lakhs (ex-showroom)
* Tiago iCNG XZ+ (Single Tone): around ₹8.60 lakhs (ex-showroom)

Please note that these prices may have changed since my knowledge cutoff, and I would recommend checking with local dealerships or the official Tata Motors website for the most up-to-date pricing information.


In [ ]:
from langchain_groq import ChatGroq
llm = ChatGroq(
    temperature=0.9, 
    model_name="llama-3.3-70b-versatile",
    
    max_tokens=500
)

In [22]:
def get_car_info(query):
    chain = RetrievalQAWithSourcesChain.from_llm(llm=llm, retriever=new_db.as_retriever())
    result = chain({"question": query}, return_only_outputs=True)
    return result

# Your specific query
query = "what is the price of Tiago iCNG?"
response = get_car_info(query)
print(response)


{'answer': 'The price of the Tiago iCNG is between Rs 6.55 lakh and Rs 8.1 lakh.\n', 'sources': 'https://www.moneycontrol.com/news/business/tata-motors-launches-punch-icng-price-starts-at-rs-7-1-lakh-11098751.html'}


In [1]:
import streamlit as st

ModuleNotFoundError: No module named 'streamlit'